# **Transformer Model Fundamentals**

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np
import math

In [ ]:
sentence = "i love ai"

vocab = {"i":0, "love":1, "ai":2}

tokens = [vocab[word] for word in sentence.split()]
tokens = np.array(tokens)[np.newaxis, :]   # batch dimension

print(tokens)


[[0 1 2]]


In [ ]:
embed_dim = 8

embedding = layers.Embedding(input_dim=len(vocab), output_dim=embed_dim)

x = embedding(tokens)
print(x.shape)


(1, 3, 8)


In [ ]:
def positional_encoding(seq_len, embed_dim):
    pe = np.zeros((seq_len, embed_dim))

    for pos in range(seq_len):
        for i in range(0, embed_dim, 2):
            pe[pos, i] = math.sin(pos / (10000 ** (i/embed_dim)))
            if i + 1 < embed_dim:
                pe[pos, i+1] = math.cos(pos / (10000 ** (i/embed_dim)))

    return tf.cast(pe, dtype=tf.float32)


In [ ]:
seq_len = x.shape[1]
x = x + positional_encoding(seq_len, embed_dim)

In [ ]:
class SelfAttention(layers.Layer):
    def __init__(self, embed_dim):
        super().__init__()
        self.embed_dim = embed_dim

        self.Q = layers.Dense(embed_dim)
        self.K = layers.Dense(embed_dim)
        self.V = layers.Dense(embed_dim)

    def call(self, x):
        Q = self.Q(x)
        K = self.K(x)
        V = self.V(x)

        scores = tf.matmul(Q, K, transpose_b=True) / tf.sqrt(tf.cast(self.embed_dim, tf.float32))
        weights = tf.nn.softmax(scores, axis=-1)

        output = tf.matmul(weights, V)
        return output


In [ ]:
class EncoderBlock(layers.Layer):
    def __init__(self, embed_dim):
        super().__init__()
        self.attention = SelfAttention(embed_dim)
        self.norm1 = layers.LayerNormalization()

        self.ffn = tf.keras.Sequential([
            layers.Dense(embed_dim, activation='relu'),
            layers.Dense(embed_dim)
        ])
        self.norm2 = layers.LayerNormalization()

    def call(self, x):
        attn = self.attention(x)
        x = self.norm1(x + attn)

        ffn_out = self.ffn(x)
        return self.norm2(x + ffn_out)


In [ ]:
class TransformerEncoder(tf.keras.Model):
    def __init__(self, embed_dim, num_layers):
        super().__init__()
        self.layers_list = [EncoderBlock(embed_dim) for _ in range(num_layers)]

    def call(self, x):
        for layer in self.layers_list:
            x = layer(x)
        return x


In [ ]:
model = TransformerEncoder(embed_dim, num_layers=2)

output = model(x)

print("Final Output Shape:", output.shape)
print(output)


Final Output Shape: (1, 3, 8)
tf.Tensor(
[[[-0.7637947   0.5788275  -0.9843865  -0.3144951  -0.5311537
   -0.73351026  2.2129936   0.535519  ]
  [-0.5651417   0.14257352 -1.3588662  -0.12901565 -0.48329428
   -0.5336526   2.1659973   0.76139957]
  [-0.34699294 -0.7905952  -1.3014424   0.06452259  0.14923123
   -0.82125646  1.8495281   1.197005  ]]], shape=(1, 3, 8), dtype=float32)
